In [ ]:
import time
import pandas as pd
import numpy as np
from datetime import datetime
from binance.client import Client
import math
from decimal import Decimal, ROUND_DOWN
import traceback

# ======================
# Binance Testnet
# ======================
API_KEY = "w1VYfTtXIx6ciJaqSupYIBM1V7DXf0VUTRZFEG8bMd82L28ynvcnFpefgGcn5QJt"
API_SECRET = "LHlbxcBmrdqe4K7aWEc5e0wh9L9Pngzca0J8uw7uSFEWSP1M59V7AfPitlFRZ8ig"
# reduce timeout to fail faster during debugging
client = Client(API_KEY, API_SECRET, requests_params={'timeout': 15})
client.FUTURES_URL = 'https://testnet.binancefuture.com/fapi'

# ======================
# 参数区（保留你的参数名与含义）
# ======================
USE_CAPITAL_RATIO = 0.75
LEVERAGE = 20
FEE_RATE = 0.0005  # 0.05% taker

ATR_MULTIPLIER = 10
HL_LOOKBACK = 20
HL_BUFFER = 1
HH_BUFFER = 1

MIN_TURNOVER = 1_000_000
SCAN_INTERVAL = 10

# TEST_NOTIONAL_CAP = None（允许按账户 & 杠杆计算），但下单会自动降级直至通过
TEST_NOTIONAL_CAP = None

# DRY_RUN: 若为 True，所有下单/取消操作只打印不发送真实请求
DRY_RUN = False

# ======================
# 辅助：网络重试
# ======================
# reduce retries during debug to avoid long blocking on network flakiness
def call_with_retry(fn, *args, retries=2, backoff=0.5, **kwargs):
    for i in range(retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            print(f"⚠️ network call failed ({i+1}/{retries}):", e)
            if i == retries - 1:
                raise
            time.sleep(backoff * (2 ** i))

# ======================
# 技术指标
# ======================
def EMA(series, period):
    return series.ewm(span=period, adjust=False).mean()

def ATR(df, period=14):
    high_low = df['high'] - df['low']
    high_close = np.abs(df['high'] - df['close'].shift())
    low_close = np.abs(df['low'] - df['close'].shift())
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period).mean()

# ======================
# 拉 K 线
# ======================
def fetch_klines(symbol, interval, limit=120):
    klines = call_with_retry(client.futures_klines, symbol=symbol, interval=interval, limit=limit)
    df = pd.DataFrame(klines, columns=[
        'ts','open','high','low','close','volume',
        '_','_','_','_','_','_'
    ])
    df = df[['ts','open','high','low','close','volume']]
    df[['open','high','low','close','volume']] = df[['open','high','low','close','volume']].astype(float)
    df['ts'] = pd.to_datetime(df['ts'], unit='ms')
    df.set_index('ts', inplace=True)
    return df

# ======================
# 基本工具
# ======================
exchange_info = {s['symbol']: s for s in call_with_retry(client.futures_exchange_info)['symbols']}

def adjust_qty(symbol, qty):
    filters = exchange_info[symbol]['filters']
    lot = next(f for f in filters if f['filterType'] == 'LOT_SIZE')
    mkt = next((f for f in filters if f['filterType'] == 'MARKET_LOT_SIZE'), None)
    step = Decimal(lot['stepSize'])
    min_qty = Decimal(lot['minQty'])
    max_qty = Decimal(lot['maxQty'])
    qty = Decimal(str(qty))
    qty = (qty // step) * step
    qty = max(qty, min_qty)
    qty = min(qty, max_qty)
    if mkt:
        mkt_max = Decimal(mkt['maxQty'])
        qty = min(qty, mkt_max)
    return float(qty)

def get_price_tick_and_precision(symbol):
    """Return (tickSize (Decimal), precision int) for price formatting."""
    f = next((f for f in exchange_info[symbol]['filters'] if f['filterType'] == 'PRICE_FILTER'), None)
    if not f:
        return Decimal('0.00000001'), 8
    tick = Decimal(f.get('tickSize'))
    prec = max(0, -tick.as_tuple().exponent) if tick != 0 else 8
    return tick, prec


def format_price_to_tick(symbol, price):
    tick, prec = get_price_tick_and_precision(symbol)
    p = Decimal(str(price))
    if tick > 0:
        p = (p // tick) * tick
    fmt = f"{0:.{prec}f}"
    return fmt.format(float(p))

def get_usdt_balance():
    try:
        bal = call_with_retry(client.futures_account_balance)
        for b in bal:
            if b.get('asset') == 'USDT':
                return float(b.get('balance', 0.0))
        return 0.0
    except Exception as e:
        print('⚠️ get_usdt_balance failed:', e)
        return 0.0

# ======================
# 计算名义与数量（返回诊断字典）
# ======================
def compute_notional_and_qty(symbol, price, usdt_balance, leverage, cap_notional=None):
    use_cap = usdt_balance * USE_CAPITAL_RATIO
    desired_notional = use_cap * leverage
    max_notional = None
    for f in exchange_info[symbol]['filters']:
        if f.get('filterType') in ('MAX_NOTIONAL', 'NOTIONAL'):
            max_notional = float(f.get('maxNotionalValue') or f.get('notional', 0) or 0)
            break
    if max_notional and max_notional > 0:
        clipped = min(desired_notional, max_notional * 0.99)
    else:
        clipped = desired_notional
    if cap_notional:
        clipped = min(clipped, cap_notional)
    raw_qty = clipped / price if price and price > 0 else 0
    adj_qty = adjust_qty(symbol, raw_qty)
    lot = next(f for f in exchange_info[symbol]['filters'] if f['filterType'] == 'LOT_SIZE')
    min_qty = float(Decimal(lot['minQty']))
    return {
        'use_cap': use_cap,
        'desired_notional': desired_notional,
        'max_notional': max_notional,
        'clipped_notional': clipped,
        'price': price,
        'raw_qty': raw_qty,
        'adj_qty': adj_qty,
        'min_qty': min_qty
    }

# ======================
# 自动降级下单（支持 dry-run）
# ======================
def place_order_with_auto_reduce(symbol, side, init_notional, price, min_notional=10.0):
    if not price or price <= 0:
        ticker = call_with_retry(client.futures_ticker, symbol=symbol)
        price = float(ticker['lastPrice'])
    notional = init_notional
    attempt = 0
    while notional >= min_notional and attempt < 3:
        qty = notional / price
        adj_qty = adjust_qty(symbol, qty)
        lot = next(f for f in exchange_info[symbol]['filters'] if f['filterType'] == 'LOT_SIZE')
        min_qty = float(Decimal(lot['minQty']))
        if adj_qty < min_qty:
            print(f"⚠️ adj_qty {adj_qty} below min_qty {min_qty}, shrinking notional")
            notional = notional * 0.9
            attempt += 1
            continue
        print(datetime.now(), f"🚀 TRY PLACING ORDER {symbol} {side} notional={notional:.2f} qty={adj_qty}")
        if DRY_RUN:
            # 模拟返回的 order 格式中的关键字段
            fake_order = {
                'orderId': int(time.time()*1000) % 1_000_000_000,
                'symbol': symbol,
                'status': 'NEW',
                'origQty': str(adj_qty)
            }
            return fake_order, notional
        try:
            order = call_with_retry(client.futures_create_order, symbol=symbol, side=side, type='MARKET', quantity=adj_qty, retries=1)
            return order, notional
        except Exception as e:
            print(f"⚠️ place attempt failed (notional={notional}):", e)
            notional = notional * 0.9
            attempt += 1
            time.sleep(0.2)
            continue
    print("⚠️ All auto-reduce attempts failed or notional too small, aborting")
    return None, 0

# ======================
# 交易所止损单：创建/取消/更新（支持 dry-run）
# 使用 STOP_MARKET + closePosition=True
# ======================
def create_exchange_stop(symbol, side, stop_price):
    """Create a STOP_MARKET closePosition order. Returns the created order dict or None on failure.

    This function formats stop_price to the symbol tick and logs the attempt.
    """
    formatted = format_price_to_tick(symbol, stop_price)
    print(f"[STOP] create stop order for {symbol} side={side} stop_price={formatted}")
    if DRY_RUN:
        return {'orderId': int(time.time()*1000) % 1_000_000_000, 'status': 'NEW', 'stopPrice': formatted}
    try:
        # For STOP_MARKET closePosition=True is recommended to close the full position
        return call_with_retry(client.futures_create_order, symbol=symbol, side=side, type='STOP_MARKET', stopPrice=str(formatted), closePosition=True)
    except Exception as e:
        # surface the exception for callers to inspect (they may try a fallback)
        print('⚠️ create_exchange_stop failed:', e)
        return None


def cancel_exchange_order(symbol, order):
    """Cancel a known order dict. If the dict lacks an orderId, try to find and cancel open conditional orders for the symbol.

    Returns True/response on success, None on failure.
    """
    if not order:
        return None

    # Prefer explicit orderId if present
    oid = order.get('orderId') or order.get('orderId') or order.get('orderIdStr')
    # Some exchange responses for conditional/algo orders may have different keys (algoId etc.).
    # If we can't determine a single id, fall back to listing open orders and cancelling matching conditional orders.
    if oid:
        print(f"[STOP] cancel order {symbol} id={oid}")
        if DRY_RUN:
            return {'status': 'CANCELED', 'orderId': oid}
        try:
            return call_with_retry(client.futures_cancel_order, symbol=symbol, orderId=oid)
        except Exception as e:
            print('⚠️ cancel_exchange_order failed for id:', oid, e)
            # fallthrough to a safer fallback below

    # Fallback: list open orders for symbol and cancel STOP_MARKET / TAKE_PROFIT_MARKET type orders
    try:
        if DRY_RUN:
            print('DRY_RUN: would list and cancel conditional open orders for', symbol)
            return None
        open_orders = call_with_retry(client.futures_get_open_orders, symbol=symbol)
        cancelled_any = False
        for o in open_orders:
            # treat STOP/TAKE_PROFIT market type and any order with triggerPrice as conditional
            otype = o.get('type', '')
            if otype in ('STOP_MARKET', 'TAKE_PROFIT_MARKET') or o.get('triggerPrice') or o.get('algoId'):
                oid2 = o.get('orderId')
                try:
                    print(f"[STOP] fallback cancelling open order {symbol} orderId={oid2} type={otype}")
                    call_with_retry(client.futures_cancel_order, symbol=symbol, orderId=oid2)
                    cancelled_any = True
                except Exception as e:
                    print('⚠️ cancel open order failed:', e)
        if cancelled_any:
            return True
        return None
    except Exception as e:
        print('⚠️ cancel_exchange_order fallback failed:', e)
        return None


def _wait_for_no_conditional_orders(symbol, timeout=1.0, poll=0.15):
    """Wait up to timeout seconds until there are no conditional open orders on the symbol.

    Returns True if no conditional orders remain, False if timeout.
    """
    if DRY_RUN:
        return True
    start = time.time()
    while time.time() - start < timeout:
        try:
            open_orders = call_with_retry(client.futures_get_open_orders, symbol=symbol)
            has_cond = False
            for o in open_orders:
                if o.get('type') in ('STOP_MARKET', 'TAKE_PROFIT_MARKET') or o.get('triggerPrice') or o.get('algoId'):
                    has_cond = True
                    break
            if not has_cond:
                return True
        except Exception as e:
            print('⚠️ _wait_for_no_conditional_orders list failed:', e)
            # continue polling until timeout
        time.sleep(poll)
    return False


def robust_update_stop(symbol, position, side_to_close, new_stop, wait_after_cancel=0.12):
    """Robustly update an exchange conditional stop: cancel existing (if any), wait for open conditional orders to clear, then create new one.

    Updates position['stop_order'] and position['stop'] on success. Returns True on success, False otherwise.
    """
    # Tick-quantize the new stop and old stop for comparison
    try:
        fmt_new = format_price_to_tick(symbol, new_stop)
        fmt_old = format_price_to_tick(symbol, position.get('stop', new_stop))
    except Exception:
        fmt_new = str(new_stop)
        fmt_old = str(position.get('stop', new_stop))

    if fmt_new == fmt_old:
        return True

    print(f"[POS] Update {position['side']} stop {symbol} old={fmt_old} new={fmt_new} (robust)")

    # Cancel existing
    try:
        cancel_exchange_order(symbol, position.get('stop_order'))
    except Exception as e:
        print('⚠️ robust_update_stop cancel error:', e)

    # small pause and wait for exchange to reflect cancellation
    time.sleep(wait_after_cancel)
    cleared = _wait_for_no_conditional_orders(symbol, timeout=1.0)
    if not cleared:
        print('[STOP] warning: conditional orders may still exist after cancel/wait')

    # Try create
    new_order = create_exchange_stop(symbol, side_to_close, new_stop)
    if new_order:
        position['stop_order'] = new_order
        position['stop'] = float(new_stop)
        return True

    # If create failed (e.g., -4130), try a final cleanup and one retry
    print('[STOP] create_exchange_stop returned None, attempting final cleanup and one retry')
    try:
        open_orders = call_with_retry(client.futures_get_open_orders, symbol=symbol)
        for o in open_orders:
            if o.get('type') in ('STOP_MARKET', 'TAKE_PROFIT_MARKET') or o.get('triggerPrice') or o.get('algoId'):
                oid2 = o.get('orderId')
                try:
                    print(f"[STOP] final cleanup cancelling open order {symbol} orderId={oid2}")
                    call_with_retry(client.futures_cancel_order, symbol=symbol, orderId=oid2)
                except Exception as e:
                    print('⚠️ final cleanup cancel failed:', e)
        time.sleep(wait_after_cancel)
        new_order = create_exchange_stop(symbol, side_to_close, new_stop)
        if new_order:
            position['stop_order'] = new_order
            position['stop'] = float(new_stop)
            return True
    except Exception as e:
        print('⚠️ robust_update_stop final cleanup failed:', e)

    print('[STOP] robust_update_stop failed to create new stop')
    # leave position['stop_order'] as None to reflect that we don't trust the exchange-side stop
    position['stop_order'] = None
    return False

# ======================
# 策略实现（基于你给的回测逻辑），现在使用交易所 stop 单管理止损
# 优化：每 T_TICK_REFRESH 秒批量刷新一次 all_tickers 并缓存 quoteVolume；主循环使用缓存避免大量 per-symbol ticker 请求
# ======================

# initial fetch of all tickers and build vol_map
try:
    all_tickers = call_with_retry(client.futures_ticker)
except Exception as e:
    print('⚠️ fetch all tickers failed:', e)
    all_tickers = []

vol_map = {}
for t in all_tickers:
    s = t.get('symbol')
    try:
        vol_map[s] = float(t.get('quoteVolume', 0) or 0)
    except Exception:
        vol_map[s] = 0.0

candidate = [s for s in exchange_info.keys() if exchange_info[s]['quoteAsset']=='USDT' and exchange_info[s]['status']=='TRADING' and exchange_info[s]['contractType']=='PERPETUAL' and s in vol_map]
sorted_syms = sorted(candidate, key=lambda s: vol_map.get(s, 0.0), reverse=True)
# 优化参数
TOP_N = 30
symbols = sorted_syms[:TOP_N]
print(f"Loaded {len(candidate)} symbols from exchange_info, selecting top {len(symbols)} by quoteVolume (TOP_N={TOP_N})")
print('Top symbols (first 20):', symbols[:20])

# Kline request limits (reduced)
KLINES_LIMIT_15 = 220
KLINES_LIMIT_1 = 50

# ticker cache refresh interval (秒)
T_TICK_REFRESH = 60
ticker_cache_time = time.time()

position = None  # 本地持仓状态

# quick timing test for first 5 symbols (keeps per-symbol ticker calls for measurement)
print('\nRunning quick timing test for first 5 symbols...')
sample = symbols[:5]
timings = {}
for i, s in enumerate(sample, 1):
    t0 = time.time()
    try:
        _ = call_with_retry(client.futures_ticker, symbol=s)
        t1 = time.time()
        df15 = fetch_klines(s, Client.KLINE_INTERVAL_15MINUTE, limit=KLINES_LIMIT_15)
        t2 = time.time()
        df1 = fetch_klines(s, Client.KLINE_INTERVAL_1MINUTE, limit=KLINES_LIMIT_1)
        t3 = time.time()
        timings[s] = {
            'ticker_ms': int((t1 - t0) * 1000),
            'kline15_ms': int((t2 - t1) * 1000),
            'kline1_ms': int((t3 - t2) * 1000),
            'total_ms': int((t3 - t0) * 1000)
        }
    except Exception as e:
        timings[s] = {'error': str(e)}
    print(f'[{i}/{len(sample)}] {s} timings:', timings[s])

print('\nSummary timings:')
print(timings)

print('\nDiagnostic done — 主循环将只遍历 top-N 符号并使用较小的 Kline 限制以加速单轮扫描')

while True:
    try:
        loop_start = time.time()
        # print(f"[LOOP] start at {datetime.now()}")
        # refresh ticker cache periodically
        if time.time() - ticker_cache_time > T_TICK_REFRESH:
            try:
                all_tickers = call_with_retry(client.futures_ticker)
                vol_map = {}
                for t in all_tickers:
                    s = t.get('symbol')
                    try:
                        vol_map[s] = float(t.get('quoteVolume', 0) or 0)
                    except Exception:
                        vol_map[s] = 0.0
                ticker_cache_time = time.time()
                # optionally re-sort symbols based on updated vol_map (keep TOP_N fixed for simplicity)
                print(f"[TICKER_CACHE] refreshed at {datetime.now()}, top sample:", list(sorted(vol_map.items(), key=lambda x: x[1], reverse=True))[:5])
            except Exception as e:
                print('⚠️ ticker cache refresh failed:', e)

        if position is None:
            for idx, sym in enumerate(symbols, 1):
                try:
                    # use cached vol_map instead of per-symbol ticker call
                    turnover = float(vol_map.get(sym, 0.0))
                    if turnover < MIN_TURNOVER:
                        continue

                    df15 = fetch_klines(sym, Client.KLINE_INTERVAL_15MINUTE, limit=KLINES_LIMIT_15)
                    df1 = fetch_klines(sym, Client.KLINE_INTERVAL_1MINUTE, limit=KLINES_LIMIT_1)

                    # 15m 因子
                    df15['EMA20'] = EMA(df15['close'], 20)
                    df15['EMA50'] = EMA(df15['close'], 50)
                    df15['EMA200'] = EMA(df15['close'], 200)
                    N = 20
                    df15['HH'] = df15['high'] > df15['high'].rolling(N).max().shift()
                    df15['HL'] = df15['low']  > df15['low'].rolling(N).min().shift()
                    df15['LL'] = df15['low']  < df15['low'].rolling(N).min().shift()
                    df15['LH'] = df15['high'] < df15['high'].rolling(N).max().shift()
                    df15['EMA20_slope'] = (df15['EMA20'] - df15['EMA20'].shift(5)) / df15['close']
                    last15 = df15.iloc[-1]

                    if (
                        last15['close'] > last15['EMA200'] and
                        last15['EMA20'] > last15['EMA50'] > last15['EMA200'] and
                        last15['EMA20_slope'] > 0.0001 #！
                    ):
                        regime = 'STRONG_UP'
                    elif (
                        last15['close'] < last15['EMA200'] and
                        last15['EMA20'] < last15['EMA50'] < last15['EMA200'] and
                        last15['EMA20_slope'] < -0.0001 #！
                    ):
                        regime = 'STRONG_DOWN'
                    else:
                        regime = 'OTHER'

                    allow_long = bool(last15.get('HH', False) and last15.get('HL', False) and last15['EMA20']>last15['EMA50'] and last15['EMA50']>last15['EMA200'])
                    allow_short = bool(last15.get('LL', False) and last15.get('LH', False) and last15['EMA20']<last15['EMA50'] and last15['EMA50']<last15['EMA200'])

                    if regime == 'STRONG_UP' and not allow_long:
                        continue
                    if regime == 'STRONG_DOWN' and not allow_short:
                        continue

                    # 1m 因子
                    df1['EMA20'] = EMA(df1['close'], 20)
                    df1['ATR'] = ATR(df1)
                    df1['VOL_SPIKE'] = df1['volume'] > 1.08 * df1['volume'].rolling(20).mean() #！
                    last1 = df1.iloc[-1]

                    # 多头入场
                    if regime == 'STRONG_UP' and allow_long and last1['low'] <= last1['EMA20'] and last1['VOL_SPIKE']:
                        usdt_balance = get_usdt_balance()
                        diag = compute_notional_and_qty(sym, last1['close'], usdt_balance, LEVERAGE, cap_notional=TEST_NOTIONAL_CAP)
                        print(f"[DEBUG] {sym} diag: {diag}")
                        if diag['adj_qty'] < diag['min_qty']:
                            continue
                        entry_price = last1['close']
                        init_stop = entry_price - ATR_MULTIPLIER * last1['ATR']
                        order, used_notional = place_order_with_auto_reduce(sym, 'BUY', diag['clipped_notional'], entry_price)
                        if order:
                            # 创建交易所止损单：使用 SELL closePosition=True
                            stop_order = create_exchange_stop(sym, 'SELL', init_stop)
                            position = {
                                'symbol': sym,
                                'side': 'long',
                                'qty': float(order['origQty']),
                                'entry_price': entry_price,
                                'stop': init_stop,
                                'used_notional': used_notional,
                                'opened_order': order,
                                'stop_order': stop_order
                            }
                            print('✅ Opened long', position)
                            break

                    # 空头入场
                    if regime == 'STRONG_DOWN' and allow_short and last1['high'] >= last1['EMA20'] and last1['VOL_SPIKE']:
                        usdt_balance = get_usdt_balance()
                        diag = compute_notional_and_qty(sym, last1['close'], usdt_balance, LEVERAGE, cap_notional=TEST_NOTIONAL_CAP)
                        print(f"[DEBUG] {sym} diag: {diag}")
                        if diag['adj_qty'] < diag['min_qty']:
                            continue
                        entry_price = last1['close']
                        init_stop = entry_price + ATR_MULTIPLIER * last1['ATR']
                        order, used_notional = place_order_with_auto_reduce(sym, 'SELL', diag['clipped_notional'], entry_price)
                        if order:
                            stop_order = create_exchange_stop(sym, 'BUY', init_stop)
                            position = {
                                'symbol': sym,
                                'side': 'short',
                                'qty': float(order['origQty']),
                                'entry_price': entry_price,
                                'stop': init_stop,
                                'used_notional': used_notional,
                                'opened_order': order,
                                'stop_order': stop_order
                            }
                            print('✅ Opened short', position)
                            break

                except Exception as e:
                    print('⚠️ per-symbol error', sym, e)
                    continue

        else:
            # 持仓管理：用交易所止损单管理（取消旧止损并下新的止损单）
            sym = position['symbol']
            # Use closed 1m candle for stop calculations to avoid micro-adjusting on the ongoing candle
            df1 = fetch_klines(sym, Client.KLINE_INTERVAL_1MINUTE, limit=HL_LOOKBACK + 5)
            # closed candle is at -2 (last is current unfinished candle at -1)
            if len(df1) < HL_LOOKBACK + 2:
                # not enough data, skip this loop
                time.sleep(0.01)
                continue
            closed = df1.iloc[-2]
            if position['side'] == 'long':
                recent_lows = df1['low'].iloc[-HL_LOOKBACK-1:-1]
                hl_stop = recent_lows.min() * HL_BUFFER
                new_stop = max(position['stop'], hl_stop)
                # use tick-quantized comparison
                fmt_new = format_price_to_tick(sym, new_stop)
                fmt_old = format_price_to_tick(sym, position.get('stop', new_stop))
                if fmt_new != fmt_old:
                    # 更新交易所止损单：取消旧单并下新单（robust）
                    robust_update_stop(sym, position, 'SELL', new_stop)
                print(f"[POS] long {sym} entry={position['entry_price']} stop={position['stop']} closed_low={closed['low']}")

                # 为了额外保险：若本地检测到触及止损但止损单未触发，仍尝试市价平仓（谨慎）
                if closed['low'] <= position['stop']:
                    print('[POS] Local stop hit detected, attempt market close')
                    if DRY_RUN:
                        print('DRY_RUN: would close long', position['qty'])
                    else:
                        try:
                            close_order = call_with_retry(client.futures_create_order, symbol=sym, side='SELL', type='MARKET', quantity=position['qty'])
                            print('🔒 Closed long via market order:', close_order)
                        except Exception as e:
                            print('⚠️ close position failed:', e)
                    # cancel stop order if exists
                    cancel_exchange_order(sym, position.get('stop_order'))
                    position = None

            else:
                recent_highs = df1['high'].iloc[-HL_LOOKBACK-1:-1]
                hh_stop = recent_highs.max() * HH_BUFFER
                new_stop = min(position['stop'], hh_stop)
                fmt_new = format_price_to_tick(sym, new_stop)
                fmt_old = format_price_to_tick(sym, position.get('stop', new_stop))
                if fmt_new != fmt_old:
                    robust_update_stop(sym, position, 'BUY', new_stop)
                print(f"[POS] short {sym} entry={position['entry_price']} stop={position['stop']} closed_high={closed['high']}")

                if closed['high'] >= position['stop']:
                    print('[POS] Local stop hit detected, attempt market close')
                    if DRY_RUN:
                        print('DRY_RUN: would close short', position['qty'])
                    else:
                        try:
                            close_order = call_with_retry(client.futures_create_order, symbol=sym, side='BUY', type='MARKET', quantity=position['qty'])
                            print('🔒 Closed short via market order:', close_order)
                        except Exception as e:
                            print('⚠️ close position failed:', e)
                    cancel_exchange_order(sym, position.get('stop_order'))
                    position = None

        loop_elapsed = time.time() - loop_start
        # print(f"[LOOP] end at {datetime.now()}, elapsed={loop_elapsed:.2f}s")
        time.sleep(SCAN_INTERVAL)

    except Exception as e:
        print('⚠️ main loop error:', e)
        time.sleep(5)


Loaded 533 symbols from exchange_info, selecting top 20 by quoteVolume (TOP_N=20)
Top symbols (first 20): ['BTCUSDT', 'ETHUSDT', 'BCHUSDT', 'XRPUSDT', 'DOGEUSDT', 'BNBUSDT', 'QUICKUSDT', 'MYROUSDT', 'VOXELUSDT', 'BIDUSDT', 'CUDISUSDT', 'PONKEUSDT', 'FISUSDT', 'REIUSDT', 'PORT3USDT', 'POWERUSDT', '1000XUSDT', 'EPTUSDT', 'TOKENUSDT', 'SIRENUSDT']

Running quick timing test for first 5 symbols...
[1/5] BTCUSDT timings: {'ticker_ms': 300, 'kline15_ms': 332, 'kline1_ms': 299, 'total_ms': 931}
[2/5] ETHUSDT timings: {'ticker_ms': 272, 'kline15_ms': 562, 'kline1_ms': 280, 'total_ms': 1115}
[3/5] BCHUSDT timings: {'ticker_ms': 283, 'kline15_ms': 543, 'kline1_ms': 550, 'total_ms': 1377}
[4/5] XRPUSDT timings: {'ticker_ms': 538, 'kline15_ms': 561, 'kline1_ms': 483, 'total_ms': 1582}
[5/5] DOGEUSDT timings: {'ticker_ms': 515, 'kline15_ms': 602, 'kline1_ms': 286, 'total_ms': 1404}

Summary timings:
{'BTCUSDT': {'ticker_ms': 300, 'kline15_ms': 332, 'kline1_ms': 299, 'total_ms': 931}, 'ETHUSDT': {'t

KeyboardInterrupt: 